# MD-GRTN Phase 1 — PEMS08
**Changes:** `num_layers=5` | `gru_layers=3` | seed fixed | checkpoint save/resume | **3-sequence MDAF (recent + 1h + 1d)**

**Architecture:** MDAF → MGRC (GCN+GRU ×3) → Spatial Transformer ×5 → Temporal Transformer ×5 → Predicted value

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════
# SEED — fixes ALL randomness across sessions
# Run this first, every single session
# ══════════════════════════════════════════════════
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} — results reproducible across sessions ✓')

set_seed()

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# ══════════════════════════════════════════════════
# CONFIG
# Changes from original:
#   num_layers : 2 → 5   (deeper spatial+temporal)
#   gru_layers : 4 → 3   (lighter MGRC)
# ══════════════════════════════════════════════════
class Config:
    # data
    data_path   = "/content/PEMS08.npz"   # change if using Drive
    num_nodes   = 170
    in_features = 3
    seq_len     = 12
    pred_len    = 12
    feature_idx = 0
    hour_offset = 12    # 1 hour ago  = 12 steps × 5 min

    # model
    d_model    = 64
    n_heads    = 4
    num_layers = 5     # ← changed from 2 to 5
    gru_layers = 3     # ← changed from 4 to 3
    dropout    = 0.1

    # training
    batch_size  = 32
    lr          = 1e-3
    epochs      = 100
    patience    = 15
    train_ratio = 0.6
    val_ratio   = 0.2

    # checkpoint
    ckpt_path   = 'mdgrtn_phase1_ckpt.pt'
    best_path   = 'mdgrtn_phase1_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config ready | num_layers={cfg.num_layers} | gru_layers={cfg.gru_layers} | device={device}')

In [ ]:
def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    print('Keys:', list(raw.keys()))
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Shape: {data.shape}')

    mean = data.mean(axis=(0,1), keepdims=True)
    std  = data.std(axis=(0,1),  keepdims=True) + 1e-8
    data = (data - mean) / std

    for key in ('adjacency_matrix','adj_mx','adj'):
        if key in raw:
            A_dist = raw[key].astype(np.float32)
            print(f'Adjacency loaded from "{key}"')
            break
    else:
        print('No adjacency found — using identity')
        A_dist = np.eye(N, dtype=np.float32)

    deg    = A_dist.sum(axis=1, keepdims=True) + 1e-8
    A_dist = A_dist / deg
    return data, mean, std, A_dist


class TrafficDataset(Dataset):
    """
    Passes the FULL data array but constrains sampling to [split_start, split_end).
    x_hour lookback indexes freely into the full array — always well-formed.
    """
    def __init__(self, data_full, seq_len, pred_len, feature_idx,
                 hour_offset=12, split_start=0, split_end=None):
        self.data        = torch.from_numpy(data_full)
        self.seq_len     = seq_len
        self.pred_len    = pred_len
        self.feat_idx    = feature_idx
        self.hour_offset = hour_offset
        T = len(data_full)
        split_end = split_end if split_end is not None else T
        # first valid i: hour lookback must not go below 0
        first_i = max(split_start, hour_offset)
        # last valid i: forward window must fit
        last_i  = split_end - pred_len
        self.indices = list(range(first_i, last_i))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x_rec  = self.data[i : i + self.seq_len]
        x_hour = self.data[i - self.hour_offset : i - self.hour_offset + self.seq_len]
        y      = self.data[i + self.seq_len : i + self.seq_len + self.pred_len,
                           :, self.feat_idx]
        return x_rec, x_hour, y


def build_dataloaders(cfg):
    # re-seed before building so shuffle is deterministic
    set_seed()
    data, mean, std, A_dist = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))

    kw = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx,
                 hour_offset=cfg.hour_offset, data_full=data)
    dl_tr = DataLoader(TrafficDataset(**ds_kw, split_start=0,  split_end=t1), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset(**ds_kw, split_start=t1, split_end=t2), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(**ds_kw, split_start=t2, split_end=T),  shuffle=False, **kw)

    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean, std, A_dist

print('Data utilities ready.')

In [ ]:
class BackNet(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, in_dim))
    def forward(self, x): return self.net(x)


class MultiHeadAttentionFusion(nn.Module):
    def __init__(self, in_dim, d_model, n_heads, n_seqs=1):
        super().__init__()
        self.projs  = nn.ModuleList([nn.Linear(in_dim, d_model) for _ in range(n_seqs)])
        self.attn   = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.fc_out = nn.Linear(d_model * n_seqs, d_model)

    def forward(self, seqs):
        B, S, N, _ = seqs[0].shape
        projected = []
        for proj, seq in zip(self.projs, seqs):
            h = proj(seq.reshape(B*S, N, -1))
            h, _ = self.attn(h, h, h)
            projected.append(h.reshape(B, S, N, -1))
        return self.fc_out(torch.cat(projected, dim=-1))


class MDModule(nn.Module):
    def __init__(self, in_features, d_model, n_seqs=1):
        super().__init__()
        self.backnets = nn.ModuleList([BackNet(in_features, d_model) for _ in range(n_seqs)])
    def forward(self, seqs):
        return [bn(s) for bn, s in zip(self.backnets, seqs)]


class MDAFModule(nn.Module):
    def __init__(self, in_features, d_model, n_heads, n_seqs=1):
        super().__init__()
        self.md  = MDModule(in_features, d_model, n_seqs)
        self.maf = MultiHeadAttentionFusion(in_features, d_model, n_heads, n_seqs)
    def forward(self, seqs):
        return self.maf(self.md(seqs))

print('MDAF defined.')

In [ ]:
class GraphAttention(nn.Module):
    """Multi-head GAT using the fused adjacency as an attention mask."""
    def __init__(self, in_dim, out_dim, n_heads=4, dropout=0.1):
        super().__init__()
        assert out_dim % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = out_dim // n_heads
        self.W       = nn.Linear(in_dim, out_dim, bias=False)
        # per-head attention vectors (size = d_head each)
        self.a_src   = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head))
        self.a_dst   = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head))
        self.drop    = nn.Dropout(dropout)
        self.out     = nn.Linear(out_dim, out_dim)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, x, A):
        # x: (B, N, in_dim)   A: (N, N)
        B, N, _ = x.shape
        h = self.W(x).reshape(B, N, self.n_heads, self.d_head)  # (B,N,H,d)
        h = h.permute(0, 2, 1, 3)                                # (B,H,N,d)
        # e_src[b,h,i,1] = (h[b,h,i,:] * a_src).sum(-1)
        e_src = (h * self.a_src).sum(-1, keepdim=True)           # (B,H,N,1)
        # e_dst[b,h,1,j] = (h[b,h,j,:] * a_dst).sum(-1)
        e_dst = (h * self.a_dst).sum(-1, keepdim=True)           # (B,H,N,1)
        # broadcast: (B,H,N,1) + (B,H,1,N) → (B,H,N,N)
        e = F.leaky_relu(e_src + e_dst.permute(0, 1, 3, 2), negative_slope=0.2)
        # mask non-edges
        mask  = (A == 0).unsqueeze(0).unsqueeze(0)               # (1,1,N,N)
        e     = e.masked_fill(mask, -1e9)
        alpha = self.drop(torch.softmax(e, dim=-1))               # (B,H,N,N)
        out   = (alpha @ h).permute(0, 2, 1, 3).reshape(B, N, -1)  # (B,N,out_dim)
        return self.out(out)


class GAT_GRU_Layer(nn.Module):
    def __init__(self, in_dim, hidden_dim, n_heads=4, dropout=0.1):
        super().__init__()
        self.gat = GraphAttention(in_dim, hidden_dim, n_heads, dropout)
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

    def forward(self, x_seq, A):
        B, S, N, _ = x_seq.shape
        h = torch.zeros(B*N, self.gru.hidden_size, device=x_seq.device)
        outs = []
        for t in range(S):
            gat_out = F.relu(self.gat(x_seq[:,t], A))       # (B,N,hidden)
            h       = self.gru(gat_out.reshape(B*N,-1), h)
            outs.append(h.reshape(B,N,-1))
        return torch.stack(outs, dim=1)


class MGRCModule(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_nodes, num_layers=3):
        super().__init__()
        self.emb      = nn.Parameter(torch.randn(num_nodes, hidden_dim))
        self.adj_conv = nn.Conv2d(2, 1, kernel_size=1)
        dims = [in_dim] + [hidden_dim]*num_layers
        self.layers = nn.ModuleList([
            GAT_GRU_Layer(dims[i], dims[i+1]) for i in range(num_layers)])

    def get_fused_adj(self, A_dist):
        A_dyna  = torch.softmax(self.emb @ self.emb.T, dim=-1)
        stacked = torch.stack([A_dist, A_dyna], dim=0).unsqueeze(0)
        A_F     = F.relu(self.adj_conv(stacked).squeeze(0).squeeze(0))
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)

    def forward(self, x, A_dist):
        A_F = self.get_fused_adj(A_dist)
        for layer in self.layers:
            x = layer(x, A_F)
        return x

print('MGRC defined with GAT (gru_layers=3).')

In [ ]:
class SpatialTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model*4), nn.ReLU(), nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, S, N, d = x.shape
        f = x.reshape(B*S, N, d)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, S, N, d)


class TemporalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model*4), nn.ReLU(), nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, S, N, d = x.shape
        f = x.permute(0,2,1,3).reshape(B*N, S, d)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, N, S, d).permute(0,2,1,3)

print('Transformer layers defined (will use 5 of each).')

In [ ]:
class MDGRTN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        L, h, dr = cfg.num_layers, cfg.n_heads, cfg.dropout
        P, S = cfg.pred_len, cfg.seq_len

        self.mdaf = MDAFModule(F, d, h, n_seqs=2)  # recent + 1h
        self.mgrc = MGRCModule(d, d, N, num_layers=cfg.gru_layers)  # gru_layers=3

        self.spatial_pos  = nn.Parameter(torch.randn(1, 1, N, d) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, S, 1, d) * 0.02)

        # num_layers=5 spatial + 5 temporal
        self.spatial_layers  = nn.ModuleList([SpatialTransformerLayer(d, h, dr)  for _ in range(L)])
        self.temporal_layers = nn.ModuleList([TemporalTransformerLayer(d, h, dr) for _ in range(L)])

        self.out_proj = nn.Linear(d * S, P)

    def forward(self, x_recent, x_hour, A_dist):
        x = self.mdaf([x_recent, x_hour])
        x = self.mgrc(x, A_dist)
        x = x + self.spatial_pos
        for layer in self.spatial_layers:
            x = layer(x)
        x = x + self.temporal_pos
        for layer in self.temporal_layers:
            x = layer(x)
        B, S, N, d = x.shape
        return self.out_proj(x.permute(0,2,1,3).reshape(B,N,S*d)).permute(0,2,1)


# Build with fixed seed
set_seed()
model = MDGRTN(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready | Parameters: {total:,}')
print(f'  num_layers (spatial+temporal): {cfg.num_layers} each')
print(f'  gru_layers                   : {cfg.gru_layers}')

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj   = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, dummy, adj)
print(f'Output shape: {out.shape}  ✓')

In [ ]:
def masked_mae(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return (torch.abs(pred-true)*mask).sum() / (mask.sum()+1e-8)

def masked_rmse(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return torch.sqrt(((pred-true)**2*mask).sum() / (mask.sum()+1e-8))

def masked_mape(pred, true, low_thresh=10.0):
    """Mask near-zero flow to avoid div/0."""
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true)/(true.abs()+1.0))*mask).sum() / mask.sum() * 100

print('Metrics defined.')

In [ ]:
# ── Mount Drive (uncomment if needed) ──
# from google.colab import drive
# drive.mount('/content/drive')
# cfg.data_path = '/content/drive/MyDrive/PEMS08.npz'

dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)

# per-sensor denorm stats
mean_flow = torch.from_numpy(mean_np[0,:,cfg.feature_idx]).to(device)  # (N,)
std_flow  = torch.from_numpy(std_np [0,:,cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)

print(f'Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}')

In [ ]:
def train_epoch(model, loader, optimizer, A_dist, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        y      = y.to(device)
        pred = model(x_rec, x_hour, A_dist)
        loss = masked_mae(pred, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, x_hour, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train/eval functions defined.')

In [ ]:
# ══════════════════════════════════════════════════
# CHECKPOINT UTILITIES
# Save to Drive at end of each session
# Resume at start of new session
# ══════════════════════════════════════════════════

def save_checkpoint(model, optimizer, scheduler, epoch, best_mae, history, cfg, drive_path=None):
    ckpt = {
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'epoch'          : epoch,
        'best_mae'       : best_mae,
        'history'        : history,
        'seed'           : SEED,
        'num_layers'     : cfg.num_layers,
        'gru_layers'     : cfg.gru_layers,
        'd_model'        : cfg.d_model,
    }
    torch.save(ckpt, cfg.ckpt_path)
    if drive_path:
        import shutil
        shutil.copy(cfg.ckpt_path, drive_path)
        print(f'Checkpoint saved to Drive: {drive_path}')
    else:
        print(f'Checkpoint saved: {cfg.ckpt_path} (epoch {epoch}, best_mae={best_mae:.3f})')


def load_checkpoint(model, optimizer, scheduler, cfg, drive_path=None):
    path = cfg.ckpt_path
    if drive_path:
        import shutil
        shutil.copy(drive_path, cfg.ckpt_path)
        print(f'Loaded from Drive: {drive_path}')
    if not os.path.exists(path):
        print('No checkpoint found — starting fresh.')
        return 1, float('inf'), {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_ep = ckpt['epoch'] + 1
    best_mae = ckpt['best_mae']
    history  = ckpt['history']
    print(f'Resumed from epoch {start_ep-1} | best_mae={best_mae:.3f} | seed={ckpt["seed"]}')
    return start_ep, best_mae, history

print('Checkpoint utilities ready.')

In [ ]:
# ══════════════════════════════════════════════════
# TRAINING
# To RESUME from checkpoint at start of new session:
#   start_epoch, best_val_mae, history = load_checkpoint(
#       model, optimizer, scheduler, cfg,
#       drive_path='/content/drive/MyDrive/mdgrtn_phase1_ckpt.pt')
# ══════════════════════════════════════════════════

set_seed()   # re-seed before training

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5, verbose=True)

# Uncomment to resume:
# start_epoch, best_val_mae, history = load_checkpoint(
#     model, optimizer, scheduler, cfg,
#     drive_path='/content/drive/MyDrive/mdgrtn_phase1_ckpt.pt')

start_epoch  = 1
best_val_mae = float('inf')
patience_cnt = 0
history      = {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}

# ── Optional: set Drive path to auto-save each epoch ──
DRIVE_CKPT = None   # set to '/content/drive/MyDrive/mdgrtn_phase1_ckpt.pt' if using Drive

print(f'Training | epochs={cfg.epochs} | patience={cfg.patience}')
print(f'Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*65)

for epoch in range(start_epoch, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)
    scheduler.step(val_mae)

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  ← best ✓'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    print(f'Ep {epoch:03d} | Loss={train_loss:.4f} | '
          f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}%{tag}')

    # Save checkpoint every epoch (survives Colab crashes)
    save_checkpoint(model, optimizer, scheduler, epoch,
                    best_val_mae, history, cfg, drive_path=DRIVE_CKPT)

print(f'\nBest Val MAE: {best_val_mae:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════
# FINAL TEST — paper-style single averaged metric
# ══════════════════════════════════════════════════
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)   # (total, 12, 170)
    Y = torch.cat(all_true, dim=0)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print('\n' + '='*55)
    print('  TEST RESULTS  —  averaged over all 12 steps')
    print('='*55)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Δ={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Δ={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Δ={mape-8.471:+.3f}%')
    print('='*55)
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(
    model, dl_test, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, x_hour, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11], ['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)